In [ ]:
import marimo as mo

import pandas as pd
import numpy as np

import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

from catboost import CatBoostClassifier

import nltk
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

In [ ]:
nltk.download('wordnet')

In [ ]:
df_train_ = pd.read_csv('data/nlp-twitter/twitter_training.csv', header=None)

In [ ]:
df_train_.columns = ["id", "topic", "sentiment", "text"]

In [ ]:
df_test_ = pd.read_csv('data/nlp-twitter/twitter_validation.csv', header=None)

In [ ]:
df_test_.columns = ["id", "topic", "sentiment", "text"]

In [ ]:
df_train_

In [ ]:
df_test_

In [ ]:
df_train_.sentiment.unique()

In [ ]:
df_train_.isna().sum()

In [ ]:
df_test_.isna().sum()

In [ ]:
df_train = df_train_.dropna()

In [ ]:
classes = df_train.sentiment.values.unique()

In [ ]:
def text_lemmatizer(text):
    lemmatizer = WordNetLemmatizer()
    return lemmatizer.lemmatize(text, pos=wordnet.VERB)

In [ ]:
df_train.sentiment = df_train.sentiment.map(lambda text: text_lemmatizer(text))

In [ ]:
df_test_.sentiment = df_test_.sentiment.map(lambda text: text_lemmatizer(text))

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1,2))

In [ ]:
X = vectorizer.fit_transform(df_train.text)

In [ ]:
y = df_train.sentiment

In [ ]:
X_val, y_val = vectorizer.transform(df_test_.text), df_test_.sentiment

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

In [ ]:
lg = LogisticRegression()

In [ ]:
lg.fit(X_train, y_train)

In [ ]:
y_pred = lg.predict(X_test)

In [ ]:
print(classification_report(y_pred, y_test))

In [ ]:
lg.fit(X, y)

In [ ]:
y_pred_val = lg.predict(X_val)

In [ ]:
print(classification_report(y_pred_val, y_val))

In [ ]:
cbc = CatBoostClassifier(max_depth=5, thread_count=-1, learning_rate=1, iterations=2000)

In [ ]:
cbc.fit(X_train, y_train)

In [ ]:
y_pred_cbc = cbc.predict(X_test)

In [ ]:
print(classification_report(y_pred_cbc, y_test))

In [ ]:
cbc.fit(X, y)

In [ ]:
y_pred_cbc_val = cbc.predict(X_val)

In [ ]:
print(classification_report(y_pred_cbc_val, y_val))

In [ ]:
# Кетбуст проиграл тупо по скорости - 20 минут обучения, против 1< минуты у LogisticRegression

print(f'Catboost, total: 20m 50s, accuracy: 95%')
print(f'LogisticRegression, total: 1< min, accuracy: 97%')